In [0]:
from pyspark.sql.functions import col, explode, from_json
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

spark.sql("CREATE CATALOG IF NOT EXISTS delta_catalog")
spark.sql("USE CATALOG delta_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS delta_demo")
spark.sql("USE SCHEMA delta_demo")

# JSON data with nested structure
json_data = [
    (1, "Rahul", '{"skills": ["PySpark", "Delta", "SQL"], "city": "Delhi"}'),
    (2, "Priya", '{"skills": ["Python", "Azure", "Databricks"], "city": "Mumbai"}'),
    (3, "Amit", '{"skills": ["Spark", "Kafka", "Scala"], "city": "Bangalore"}')
]

json_df = spark.createDataFrame(json_data, ["emp_id", "name", "details"])

# Parse JSON
json_schema=StructType([StructField("skills",ArrayType       (StringType())),
                        StructField("city",StringType())
                         ])

json_df.display()
                           
parsed_df=json_df.withColumn("details_parsed",from_json(col("details"),json_schema))
parsed_df.select("details_parsed.*","details_parsed.skills").display()

#EXPLODE SKILLS ARRAY
exploded_df=parsed_df.select("emp_id","name",col("details_parsed.city").alias("city"),
                 explode(col("details_parsed.skills")).alias("skills"))
exploded_df.display()

exploded_df.write.format("delta").mode("overwrite").saveAsTable("delta_catalog.delta_demo.employee_skills")

print(" JSON Explode Done!")
exploded_df.show()
                 
#

In [0]:
# CRUD - Create Read Update Delete on Delta Table

# CREATE
crud_data = [
    (1, "Laptop", 75000, "Electronics"),
    (2, "Phone", 25000, "Electronics"),
    (3, "Shirt", 1500, "Clothing"),
    (4, "Shoes", 5000, "Clothing"),
    (5, "Tablet", 45000, "Electronics")
]
crud_df = spark.createDataFrame(crud_data, ["id", "product", "price", "category"])
crud_df.write.format("delta").mode("overwrite").saveAsTable("delta_catalog.delta_demo.products_crud")
print(" CREATE Done!")

# READ
print("\n READ:")
spark.sql("SELECT * FROM delta_catalog.delta_demo.products_crud").show()

# UPDATE
spark.sql("""
    UPDATE delta_catalog.delta_demo.products_crud
    SET price = 80000
    WHERE id = 1
""")
print(" UPDATE Done — Laptop price updated to 80000!")
spark.sql("SELECT * FROM delta_catalog.delta_demo.products_crud WHERE id=1").show()

# DELETE
spark.sql("""
    DELETE FROM delta_catalog.delta_demo.products_crud
    WHERE category = 'Clothing'
""")
print(" DELETE Done — Clothing products removed!")
spark.sql("SELECT * FROM delta_catalog.delta_demo.products_crud").show()

In [0]:
from pyspark.sql.functions import sum

# Sales data for pivot
pivot_data = [
    ("Electronics", "Q1", 150000),
    ("Electronics", "Q2", 200000),
    ("Electronics", "Q3", 180000),
    ("Clothing", "Q1", 50000),
    ("Clothing", "Q2", 75000),
    ("Clothing", "Q3", 60000),
    ("Accessories", "Q1", 30000),
    ("Accessories", "Q2", 45000),
    ("Accessories", "Q3", 35000)
]

pivot_df = spark.createDataFrame(pivot_data, ["category", "quarter", "revenue"])

# PIVOT — quarters become columns
result = pivot_df.groupBy("category") \
    .pivot("quarter") \
    .agg(sum("revenue"))

result.write.format("delta").mode("overwrite").saveAsTable("delta_catalog.delta_demo.sales_pivot")

print("Pivot Done!")
result.show()

**Partitioning**

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

data = [
    (1, "2025-01-15", "Delhi",   "Electronics", 75000),
    (2, "2025-01-20", "Mumbai",  "Electronics", 45000),
    (3, "2025-02-10", "Delhi",   "Clothing",     2500),
    (4, "2025-02-15", "Bangalore","Grocery",     1200),
    (5, "2025-03-05", "Mumbai",  "Clothing",     3500),
    (6, "2025-03-12", "Delhi",   "Grocery",       800),
    (7, "2025-04-08", "Chennai", "Electronics", 60000),
    (8, "2025-04-22", "Bangalore","Clothing",    1800),
]

df = spark.createDataFrame(data, ["id","order_date","city","category","amount"]) \
         .withColumn("order_date", to_date("order_date")) \
         .withColumn("year",  year("order_date")) \
         .withColumn("month", month("order_date"))

df.show()
# print("Default partitions:", df.rdd.getNumPartitions())

print(f"No of partition :{df.select(spark_partition_id()).distinct().count()}")


In [0]:
#  Table Partitioning
df.write.format("delta")\
    .mode("overwrite")\
        .partitionBy("year","month")\
             .saveAsTable("delta_catalog.delta_demo.sales_partitioned")
   

# Partition Pruning
result=spark.table("delta_catalog.delta_demo.sales_partitioned")\
    .where("year=2025 and month=4")

result.explain(True) # look for PartitionFilters in the plan
result.show()

In [0]:
%sql
-- See the physical files Delta wrote
DESCRIBE DETAIL delta_catalog.delta_demo.sales_partitioned;

In [0]:
%sql
-- Look at partition folders
SHOW PARTITIONS delta_catalog.delta_demo.sales_partitioned;

In [0]:
%sql
-- Pruned query — Spark scans 1 partition
EXPLAIN SELECT * FROM delta_catalog.delta_demo.sales_partitioned 
WHERE year = 2025 AND month = 2;

In [0]:
%sql
-- Non-pruned query — Spark scans ALL partitions
EXPLAIN SELECT * FROM delta_catalog.delta_demo.sales_partitioned 
WHERE amount > 50000;

### repartition() vs coalesce() — In-Memory Partitions
 
#### MAGIC Reset point: we're now talking about how Spark splits data across executors **while processing**, not how files are laid out on disk.


In [0]:
# Reload from the Delta table
df = spark.table("delta_catalog.delta_demo.sales_partitioned")

from pyspark.sql.functions import spark_partition_id

def num_partitions(df):
    return df.select(spark_partition_id()).distinct().count()

def show_partition_distribution(df):
    df.groupBy(spark_partition_id().alias("partition_id"))\
        .count()\
            .orderBy("partition_id")\
                .show()
print(f"initial partition:{num_partitions(df)}")
show_partition_distribution(df)


  ### repartition() — full shuffle, can go up OR down



In [0]:
# Increase partitions
df_8 = df.repartition(8)

print(num_partitions(df_8))

# Serverless-safe alternative doesnt show empty partitions 
from pyspark.sql.functions import spark_partition_id, lit
show_partition_distribution(df_8)



In [0]:
# Repartition by column → rows with same city land together
df_by_city=df.repartition("city")
print("\n After repartition ('city'):",num_partitions(df_by_city))
show_partition_distribution(df_by_city)

In [0]:
big_by_city = df.repartition(5, "city")  # exact 5 partitions, by city
print("Partitions:", num_partitions(big_by_city))

big_by_city.select("city", spark_partition_id().alias("pid")) \
           .groupBy("pid", "city").count() \
           .orderBy("pid").show()

In [0]:
# Lesson 1 — Same-key co-location works:
# All 3 Delhi rows landed in partition 4. All 2 Mumbai rows landed in partition 1. Same city → same partition. ✅ That's the guarantee Spark gives you with repartition("col"), and it's the foundation of efficient joins later.
# Lesson 2 — Hash collisions are real:
# You asked for 5 partitions and 4 cities, but only 3 non-empty partitions appeared. Chennai and Bangalore both hashed into partition 0 (collision), and partitions 2 and 3 stayed empty.
# Hash function: hash(city) % 5 → if two different cities produce the same modulo, they share a partition. This is normal — and it's why interviewers ask "what if your data is skewed?" — because hash collisions on a skewed key make it worse.

In [0]:
# Evidence journal entry for today:

# "Spark's repartition(n, col) hashes the column value to assign partitions. Same value always lands in the same partition (guaranteed), but different values may share a partition due to hash collisions. Empty partitions are expected with low cardinality."



In [0]:
# Bigger dataset — 5 cities, 50k rows total
from pyspark.sql.functions import col, when, rand

big_df = spark.range(0, 50000) \
    .withColumn("city", 
        when(col("id") % 5 == 0, "Delhi")
        .when(col("id") % 5 == 1, "Mumbai")
        .when(col("id") % 5 == 2, "Bangalore")
        .when(col("id") % 5 == 3, "Chennai")
        .otherwise("Kolkata")
    ) \
    .withColumn("amount", (rand() * 10000).cast("int"))

big_by_city = big_df.repartition(5, "city")
print("Partitions:", num_partitions(big_by_city))

big_by_city.select("city", spark_partition_id().alias("pid")) \
           .groupBy("pid", "city").count() \
           .orderBy("pid").show()

In [0]:
# Three big lessons from this single output — write these in your evidence journal:

# Same-key co-location guarantee holds. All 10,000 Delhi rows are in one partition. All 10,000 Mumbai rows in one partition. This is what makes shuffle-free joins possible.
# Hash collisions are unavoidable with low cardinality. 5 cities into 5 slots looks like it should be 1-per-slot, but hash functions don't distribute small input spaces evenly. With higher cardinality (say 1000 cities into 200 partitions), the distribution would smooth out.
# This creates skew. Partitions 0 and 4 have 20k rows each; partition 1 has 10k; partitions 2 and 3 have 0. If you ran a groupBy next, the executors holding partitions 0 and 4 would do 2x the work of the one holding partition 1. That's data skew — a top-3 interview question.


# "If I asked Spark for 5 partitions on 5 unique values and got 3 non-empty ones with uneven sizes, that's hash collision causing skew. In real work, I'd either increase the partition count (repartition(20, "city") gives the hash function more room to spread values out), or use salting for severely skewed keys."

In [0]:
# 5 cities, but ask for 20 partitions — gives hash more room to spread
df_spread = big_df.repartition(20, "city")
print("Non-empty partitions:", num_partitions(df_spread))

df_spread.select("city", spark_partition_id().alias("pid")) \
         .groupBy("pid", "city").count() \
         .orderBy("pid").show(20)

In [0]:
from pyspark.sql.functions import col, concat, lit, spark_partition_id

# Generate 1 million rows with 1000 unique cities
huge_df = spark.range(0, 1000000) \
    .withColumn("city", concat(lit("City_"), (col("id") % 1000).cast("string"))) \
    .withColumn("amount", (col("id") * 7 % 10000))

print("Total rows:", huge_df.count())
print("Unique cities:", huge_df.select("city").distinct().count())

# Repartition by city into 200 partitions
huge_by_city = huge_df.repartition(200, "city")
print("Non-empty partitions:", num_partitions(huge_by_city))

In [0]:
# Row counts per partition
huge_by_city.groupBy(spark_partition_id().alias("pid")) \
            .count() \
            .orderBy("count") \
            .show(5)   # 5 smallest partitions

huge_by_city.groupBy(spark_partition_id().alias("pid")) \
            .count() \
            .orderBy(col("count").desc()) \
            .show(5)   # 5 biggest partitions

In [0]:
# Your output:

# Total rows: 1,000,000
# Unique cities: 1,000
# Non-empty partitions: 197 out of 200 requested
# Smallest partitions: 1,000 rows each (one city alone in a partition)
# Biggest partitions: 11,000 rows (partition 166), then four with 10,000, one with 9,000


# Decoding what happened:
# Each city has exactly 1,000 rows. So:
# Rows in partitionWhat it means1,0001 city alone there9,0009 cities collided10,00010 cities collided11,00011 cities collided ← the most crowded one
# Most partitions hold around 5 cities (the expected average of 1000÷200). The range is roughly 1 to 11 cities per partition, with the vast majority clustering around 5.
# 3 partitions out of 200 ended up empty — that's a 1.5% miss rate. Compare that to your 5-cities-into-5-partitions case where 40% of slots were empty.

# The comparison table — the actual lesson:
# SetupEmpty partitionsSkew (max/min ratio)Verdict5 cities → 5 slots2 of 5 (40%)infinite (20k vs 0)unusable1000 cities → 200 slots3 of 200 (1.5%)11x (11k vs 1k)acceptable
# This is the law of large numbers in production. Same hash function. Same Spark engine. The only thing you changed was cardinality. Random distribution feels "rigged" with small inputs and "fair" with large ones.

# Run the stddev cell to put a number on it:
# pythonfrom pyspark.sql.functions import min as _min, max as _max, avg, stddev

# stats = huge_by_city.groupBy(spark_partition_id().alias("pid")).count()

# stats.agg(
#     _min("count").alias("min_rows"),
#     _max("count").alias("max_rows"),
#     avg("count").alias("avg_rows"),
#     stddev("count").alias("stddev_rows")
# ).show()
# You'll likely see something like:
# min=1000, max=11000, avg=~5076, stddev=~1500
# That stddev ÷ avg ≈ 0.3 → meaning typical partitions vary about 30% from the average. That's tolerable. For perfect partitioning you'd want stddev/avg below 0.1, which is rare without explicit work (salting, range partitioning).

# The interview-ready paragraph — practice saying this out loud:

# "When I repartition by a column with high cardinality, hash collisions still happen but they distribute evenly because of the law of large numbers. With 1000 cities into 200 partitions, I got 1.5% empty partitions and a max/min ratio of 11x — acceptable for most workloads. With 5 cities into 5 partitions, I got 40% empties and infinite skew because the hash function had no room to spread values. The fix at low cardinality is either to over-partition (give hash more slots), or to switch from hash to range partitioning, or to salt the key if specific values are hot."

# That paragraph alone covers a 5-minute interview answer.

# Solid stopping point. You've now finished Partitioning/Repartition/Coalesce with real intuition, not just syntax. Add this to your evidence journal:

# " I pushed through 2 deep-dive topics (Window Functions + Partitioning/Repartition/Coalesce). Debugged 4 serverless errors and turned each into a real learning moment. Proof of capability: I can now explain hash collision and AQE coalescing from my own notebook output, not from memorizing a blog post."


 ###  coalesce() — no shuffle, can only go DOWN

In [0]:
 

# Start with 8 partitions, collapse to 2
df_collapsed = df.repartition(8).coalesce(2)
print("After repartition(8).coalesce(2):", num_partitions(df_collapsed))
show_partition_distribution(df_collapsed)

# Try to "increase" with coalesce — it won't
df_no_increase = df.coalesce(20)
print("\nAfter coalesce(20) on small df:", num_partitions(df_no_increase))
show_partition_distribution(df_no_increase)
# Stays at the original count or lower — coalesce CANNOT increase
# What to notice:

# Row counts after coalesce are often uneven — that's because it merges existing partitions locally without redistribution.
# coalesce(20) didn't go up. Coalesce can only reduce.




### The classic use case for coalesce — writing a single output file

In [0]:


# Filter heavily, then write as one CSV
small_result = df.filter("category = 'Grocery'").coalesce(1)
print("Partitions before write:", num_partitions(small_result))

# In real work you'd write to a UC volume or external location:
# small_result.write.mode("overwrite").csv("/Volumes/<catalog>/<schema>/<vol>/grocery_report")

small_result.show()
#  Never do coalesce(1) on a large DataFrame — all data funnels to one executor → OOM.

# COMMAND ----------
# MAGIC %md ## 4. Side-by-Side Summary
# MAGIC 
# MAGIC | Aspect | `repartition(n)` | `coalesce(n)` |
# MAGIC |---|---|---|
# MAGIC | Shuffle? | **Yes (full)** | **No (local merge)** |
# MAGIC | Can increase partitions? | ✅ Yes | ❌ No |
# MAGIC | Can decrease partitions? | ✅ Yes | ✅ Yes |
# MAGIC | Result partition sizes | Even | Possibly uneven |
# MAGIC | Cost | High | Low |
# MAGIC | Typical use | Before joins/groupBy, fixing skew | Before final write, after heavy filter |

# COMMAND ----------
# MAGIC %md ## 5. Interview Gotchas — Quick Recall
# MAGIC 
# MAGIC 1. **Disk partitioning vs in-memory partitions** → independent concepts. Table partitioned by `year` on disk can be read into Spark with 200 in-memory partitions.
# MAGIC 2. **Default `spark.sql.shuffle.partitions`** → 200. Tune so each shuffle partition is ~128 MB.
# MAGIC 3. **`coalesce(1)` OOM** → all data on one executor.
# MAGIC 4. **`repartition` always shuffles**, even if `n` equals current partition count.
# MAGIC 5. **`coalesce` can never increase** partitions — common trap question.
# MAGIC 6. **AQE** (Adaptive Query Execution) auto-coalesces shuffle partitions post-shuffle if enabled (`spark.sql.adaptive.coalescePartitions.enabled=true`). We'll cover AQE in detail later in the deep dive.


In [0]:
# Step 5: Self-check — answer these two without scrolling up:

# You have a 500 GB sales table queried daily by country and event_date. How would you partition it on disk? Why?
# After a heavy .filter(), your DataFrame has 200 partitions but only ~10 MB total. Do you call repartition(4) or coalesce(4)? Why?

In [0]:
# If you have ~200 countries × ~365 days × multiple years = millions of tiny folders. That's the small files problem we discussed earlier — kills query planning.
# Better answer:

# "Partition only by event_date (or even better, event_month) because date is almost always in the WHERE clause for time-series data. Skip country as a partition column because it's high cardinality and would multiply folder count. Instead, use Z-ORDER (or Liquid Clustering on newer Delta) on country — it gives you data-skipping benefits without exploding the folder count."
# sqlOPTIMIZE sales ZORDER BY (country);

# Rule of thumb to memorize: Partition by ONE low-cardinality, frequently-filtered column (usually a date column). Use Z-ORDER for the second filter column. We'll cover Z-ORDER in detail when we hit the Optimize/Vacuum/Z-Order topic.

# Question 2 — your answer:

# "coalesce(4) — small data, many empty partitions, 128 MB per partition target, coalesce is sufficient."

# Verdict: Correct. 
# You hit all three key points:

# Data is small (10 MB total)
# 200 partitions for 10 MB is wasteful (mostly empty)
# Coalesce avoids the shuffle cost

# One more sentence to make it complete:

# "I'd use coalesce(4) because reducing partitions doesn't need a shuffle — coalesce merges existing partitions locally, which is cheaper than repartition(4). Repartition would correctly give me 4 evenly-sized partitions, but the full shuffle cost isn't worth it for only 10 MB of data."

# The trade-off you should know: coalesce(4) may give slightly uneven partition sizes (e.g., 3MB, 2MB, 4MB, 1MB) but it's cheap. repartition(4) gives perfectly even (~2.5MB each) but pays for a shuffle. For 10MB → coalesce wins, easily.


In [0]:
# Cardinality = the number of distinct values in a column.


# The interview-ready rule:

# "For disk partitioning, pick low cardinality columns that appear in WHERE clauses — typically a date column. For Z-ORDER or hash repartitioning before joins, pick high cardinality columns. The intuition: partition columns create folders (so you want few), and clustering/hash columns need to spread data evenly (so you want many distinct values)."


# One-liner to memorize:

# Partition by low cardinality. Cluster by high cardinality.

# That single rule answers 80% of Spark partitioning questions interviewers throw at you.
# Got it? Or want me to walk through how this applies to a specific table you might design?Opus 4.7

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC # Broadcast Join
# MAGIC 
# MAGIC **The problem broadcast join solves:** regular joins require a shuffle — both tables get repartitioned by the join key across the network. For a big fact table + small dimension table, this is wasteful: why shuffle a 100GB table just to join it with a 5MB lookup?
# MAGIC 
# MAGIC **The idea:** send the small table to **every executor** (broadcast it), then join locally. No shuffle of the big table.
# MAGIC 
# MAGIC ## When to use
# MAGIC - One side is small (default threshold: 10 MB, configurable up to ~1 GB)
# MAGIC - Typically: fact table ⋈ dimension table
# MAGIC - Star schema queries — the classic use case
# MAGIC 
# MAGIC ## When NOT to use
# MAGIC - Both tables are large → broadcasting will OOM the driver
# MAGIC - Small table is still hundreds of MB and joined infrequently → cost > benefit

In [0]:
# Cell 1 — Build a big fact table + small dim table
from pyspark.sql.functions import col, expr, broadcast, rand

# Fact table: 1 million orders
orders_big = spark.range(0, 1_000_000) \
    .withColumnRenamed("id", "order_id") \
    .withColumn("product_id", (col("order_id") % 100).cast("int")) \
    .withColumn("amount", (rand() * 5000).cast("int"))

# Dim table: 100 products
products_small = spark.range(0, 100) \
    .withColumnRenamed("id", "product_id") \
    .withColumn("product_name", expr("concat('Product_', product_id)"))

print("Orders rows:", orders_big.count())
print("Products rows:", products_small.count())

In [0]:
# Cell 2 — Regular join (the slow way conceptually)
# regular = orders_big.join(products_small, "product_id")
# regular.show()
# print("=== PLAN ===")

# Cell 2 — FORCE SHUFFLE join using hint
shuffle_join = orders_big.hint("shuffle_merge").join(products_small, "product_id")

print("=== SHUFFLE JOIN PLAN ===")
shuffle_join.explain()
# regular.explain()


# Look for: word SortMergeJoin or Exchange in the plan. (On serverless with AQE, you might actually see BroadcastHashJoin already because the data is tiny — that's fine, just note it.)

In [0]:
# Cell 3 — Broadcast join (explicit)
fast = orders_big.join(broadcast(products_small), "product_id")
fast.show()
print("=== PLAN ===")
fast.explain()

In [0]:
# Cell 4 — Time them to feel the difference
import time

# Shuffle join
start = time.time()
shuffle_join.count()
print(f"Shuffle join: {time.time() - start:.2f} seconds")

# Broadcast join
start = time.time()
fast.count()
print(f"Broadcast join: {time.time() - start:.2f} seconds")

In [0]:
# "I tested broadcast vs shuffle join on 1 million rows with a 100-row dimension table. Shuffle join produced two Exchange hashpartitioning operations — both sides moved across the network — and ran in 1.12 seconds, also falling back from Photon to JVM because SortMergeJoin isn't fully Photon-supported. Broadcast join produced one BroadcastExchange on the small side only, ran fully on Photon, and finished in 0.50 seconds — 2.2x faster. In production with billions of rows, this gap becomes 20–50x."

# That's a complete, evidence-backed interview answer. Most candidates say "broadcast is faster because the small table goes to each executor." You can say that AND show the plan AND quote timing numbers from your own notebook.


In [0]:
# python# Cell 1 — Build a big fact table + small dim table
# from pyspark.sql.functions import col, expr, broadcast, rand

# # Fact table: 1 million orders
# orders_big = spark.range(0, 1_000_000) \
#     .withColumnRenamed("id", "order_id") \
#     .withColumn("product_id", (col("order_id") % 100).cast("int")) \
#     .withColumn("amount", (rand() * 5000).cast("int"))

# # Dim table: 100 products
# products_small = spark.range(0, 100) \
#     .withColumnRenamed("id", "product_id") \
#     .withColumn("product_name", expr("concat('Product_', product_id)"))

# print("Orders rows:", orders_big.count())
# print("Products rows:", products_small.count())

# python# Cell 2 — FORCE SHUFFLE join using hint
# shuffle_join = orders_big.hint("shuffle_merge").join(products_small, "product_id")

# print("=== SHUFFLE JOIN PLAN ===")
# shuffle_join.explain()
# Look for these keywords in the plan:
# SortMergeJoin
# ├── Exchange hashpartitioning(product_id, ...)   ← shuffles 1M orders
# │   └── ... orders_big ...
# └── Exchange hashpartitioning(product_id, ...)   ← shuffles 100 products
#     └── ... products_small ...
# Two Exchange operations = both sides shuffled across the network. The 1-million-row orders table gets repartitioned. Expensive.

# python# Cell 3 — BROADCAST join
# broadcast_join = orders_big.join(broadcast(products_small), "product_id")

# print("=== BROADCAST JOIN PLAN ===")
# broadcast_join.explain()
# Look for these keywords:
# BroadcastHashJoin
# ├── ... orders_big ...                ← stays put, NO shuffle!
# └── BroadcastExchange ...
#     └── ... products_small ...        ← only the small side moves
# Just ONE BroadcastExchange (on the small side). The 1-million-row table never moved. That's the win.

# python# Cell 4 — Time them to feel the difference
# import time

# # Shuffle join
# start = time.time()
# shuffle_join.count()
# print(f"Shuffle join: {time.time() - start:.2f} seconds")

# # Broadcast join
# start = time.time()
# broadcast_join.count()
# print(f"Broadcast join: {time.time() - start:.2f} seconds")
# Broadcast should be noticeably faster — maybe 2-5x on this size. On real production data (100GB fact + 5MB dim), the gap becomes 10-100x.

# What to paste back to me:

# The two .explain() outputs (Cells 2 and 3)
# The two timing numbers (Cell 4)

# Then you'll have seen and measured both join strategies on the same data. That's an interview story you can tell with conviction:

# "I tested both on 1M rows. Shuffle join produced two Exchange operations and ran in X seconds. Broadcast join produced one BroadcastExchange (only the small side moved) and ran in Y seconds — roughly Z times faster."

In [0]:
# COMMAND ----------
# MAGIC %md ## 2. Schema Enforcement — Delta REJECTS mismatched writes
# MAGIC
# MAGIC This is ON by default. You cannot turn it off.

# New data with an EXTRA column "department"
new_data_extra_col = spark.createDataFrame([
    (4, "Vikram", 85000, "Engineering"),
    (5, "Sneha",  70000, "Marketing"),
], ["id", "name", "salary", "department"])

# This will FAIL — Delta enforces the existing schema
try:
    new_data_extra_col.write.format("delta") \
        .mode("append") \
        .saveAsTable("delta_catalog.delta_demo.employees")
except Exception as e:
    print("ENFORCEMENT KICKED IN:")
    print(str(e)[:300])  # first 300 chars of error